6.0 Rerun baseline na curated_v2

Učitava se Data/curated_v2, trenira se kratko i meri se test accuracy i macro F1. Rezultati se snimaju u Runs/rerun_v2_YYYYMMDD_HHMMSS.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, math
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

ROOT = Path("/content/drive/MyDrive/Diplomski")
CURATED_V2 = ROOT / "Data" / "curated_v2"
RUNS = ROOT / "Runs"
RUNS.mkdir(parents=True, exist_ok=True)

run_id = time.strftime("%Y%m%d_%H%M%S")
OUT = RUNS / f"rerun_v2_{run_id}"
OUT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("OUT:", OUT)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def load_meta(ds):
    with open(CURATED_V2 / ds / "meta.json", "r", encoding="utf-8") as f:
        return json.load(f)

def load_split(ds, split):
    return pd.read_csv(CURATED_V2 / ds / f"{split}.csv")

class ImageCsvDataset(torch.utils.data.Dataset):
    def __init__(self, df, tfm=None, label_to_idx=None):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.pcol = "path"
        self.ycol = "label"
        labels = self.df[self.ycol].astype(str).tolist()
        if label_to_idx is None:
            uniq = sorted(list(set(labels)))
            self.label_to_idx = {u:i for i,u in enumerate(uniq)}
        else:
            self.label_to_idx = label_to_idx
        self.y = [self.label_to_idx[str(x)] for x in labels]

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        p = str(self.df.iloc[i][self.pcol])
        y = self.y[i]
        im = Image.open(p).convert("RGB")
        if self.tfm: im = self.tfm(im)
        return im, torch.tensor(y, dtype=torch.long)

@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        pred = torch.argmax(model(x), dim=1)
        ys += y.cpu().numpy().tolist()
        ps += pred.cpu().numpy().tolist()
    return float(accuracy_score(ys, ps)), float(f1_score(ys, ps, average="macro")), ys, ps

def train_one_epoch(model, loader, opt, loss_fn):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")

def build_resnet18(num_classes):
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def rerun_v2(ds, seed=42, img_size=224, batch_size=32, epochs=1, lr=3e-4, weight_decay=1e-2):
    set_seed(seed)

    df_tr = load_split(ds, "train")
    df_va = load_split(ds, "val")
    df_te = load_split(ds, "test")

    tfm_tr = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
    ])
    tfm_ev = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])

    ds_tr = ImageCsvDataset(df_tr, tfm=tfm_tr, label_to_idx=None)
    ds_va = ImageCsvDataset(df_va, tfm=tfm_ev, label_to_idx=ds_tr.label_to_idx)
    ds_te = ImageCsvDataset(df_te, tfm=tfm_ev, label_to_idx=ds_tr.label_to_idx)

    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    dl_te = torch.utils.data.DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    model = build_resnet18(len(ds_tr.label_to_idx)).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    hist = []
    best = {"val_f1": -1, "state": None}

    for ep in range(1, epochs+1):
        tr_loss = train_one_epoch(model, dl_tr, opt, loss_fn)
        va_acc, va_f1, _, _ = eval_loader(model, dl_va)
        hist.append({"epoch": ep, "train_loss": tr_loss, "val_acc": va_acc, "val_f1_macro": va_f1})
        if va_f1 > best["val_f1"]:
            best["val_f1"] = va_f1
            best["state"] = {k: v.detach().cpu() for k, v in model.state_dict().items()}

        print(ds, "epoch", ep, "loss", tr_loss, "val_acc", va_acc, "val_f1", va_f1)

    if best["state"] is not None:
        model.load_state_dict({k: v.to(device) for k, v in best["state"].items()})

    te_acc, te_f1, y_true, y_pred = eval_loader(model, dl_te)
    cm = confusion_matrix(y_true, y_pred)

    res = {
        "dataset": ds,
        "seed": seed,
        "img_size": img_size,
        "batch_size": batch_size,
        "epochs": epochs,
        "test_acc": te_acc,
        "test_f1_macro": te_f1,
        "label_to_idx": ds_tr.label_to_idx,
        "history": hist,
        "confusion_matrix": cm.tolist()
    }
    return res

results = []
for ds in ["lc25000", "rm1000_lung_history"]:
    r = rerun_v2(ds, seed=42, epochs=1, batch_size=32, img_size=224)
    results.append(r)

df = pd.DataFrame([{
    "dataset": r["dataset"],
    "test_acc": r["test_acc"],
    "test_f1_macro": r["test_f1_macro"]
} for r in results])
df.to_csv(OUT / "rerun_v2_summary.csv", index=False)

for r in results:
    d = OUT / r["dataset"]
    d.mkdir(parents=True, exist_ok=True)
    with open(d / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(r, f, ensure_ascii=False, indent=2)

print("Saved:", OUT / "rerun_v2_summary.csv")
df

Mounted at /content/drive
device: cuda
Tesla T4
OUT: /content/drive/MyDrive/Diplomski/Runs/rerun_v2_20260318_180611
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 99.9MB/s]


lc25000 epoch 1 loss 0.006023776890854143 val_acc 1.0 val_f1 1.0
rm1000_lung_history epoch 1 loss 0.10009820740471811 val_acc 0.9547270306258322 val_f1 0.9548217001977791
Saved: /content/drive/MyDrive/Diplomski/Runs/rerun_v2_20260318_180611/rerun_v2_summary.csv


,dataset,test_acc,test_f1_macro
0,lc25000,1.000000,1.000000
1,rm1000_lung_history,0.957257,0.957255
